In [33]:
import logging
logging.getLogger("pdfminer").setLevel(logging.ERROR)

import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "catcher-rag-eval"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [34]:
import os
from pathlib import Path
import sys

def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: c:\Users\user\catcher-llm


# 1. 문서 로드

In [35]:
PDF_PATHS = [
    PROJECT_ROOT / "data" / "raw" / "pdf" / "welfare" / "2026_hope_ladder_selected.pdf",
]

In [36]:
from langchain_community.document_loaders import PDFPlumberLoader, TextLoader

all_docs = []

for path in PDF_PATHS:
    if str(path).endswith(".pdf"):
        loader = PDFPlumberLoader(str(path))
    elif str(path).endswith(".txt"):
        loader = TextLoader(str(path), encoding="utf-8")
    else:
        continue

    docs = loader.load()

    for d in docs:
        d.metadata["source"] = path.name

    all_docs.extend(docs)

print("총 문서 수:", len(all_docs))
print(all_docs[0].page_content[:300])

총 문서 수: 49
모두의 정책 K-희망사다리 2026 037
여성청소년 성평등가족부 청소년정책과
02-2100-6242
생리용품 지원
지원대상 • 기초생활수급(생계·의료·주거·교육급여), 법정차상위계층, 한부모가족 지원
대상 가구의 9~24세 여성청소년
핵심내용 •여 성청소년 생리용품 바우처 지원(월 1만 4,000원), 국민행복카드로 구매
•9세가 되는 해의 1월 1일부터 24세가 끝나는 해의 12월 31일까지 지원
이용방법 •온 라인 신청: 복지로(www.bokjiro.go.kr) 또는 모바일 앱
•방문 신청: 읍·면·동 주민센터 및 행정복지센터


# 2. 문서 split

In [37]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

split_docs = text_splitter.split_documents(all_docs)

print(f"청킹 후 문서 수: {len(split_docs)}")

청킹 후 문서 수: 57


# 3. 임베딩 + 벡터 DB

In [38]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(split_docs, embeddings)

print(f"벡터 수: {vectorstore.index.ntotal}")

벡터 수: 57


# 4. Retriever 설정

In [39]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

# 5. 질문 → context → 답변 생성

In [40]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=300)

def run_rag(q):
    docs = retriever.invoke(q)

    context_texts = [doc.page_content[:400] for doc in docs]

    answer = llm.invoke(
        f"""
질문: {q}

아래 문서에 있는 내용만 사용해서 핵심 답변을 1~2문장으로 작성하세요.
문서에 없는 내용은 절대 추가하지 마세요.

문서:
{context_texts}
"""
    ).content

    sources = [doc.metadata.get("source", "출처 없음") for doc in docs]

    return answer, context_texts, sources

In [41]:
# 단건 테스트
q = "청년의 소비습관에 관해 알려줘."

answer, contexts, sources = run_rag(q)

print("답변:")
print(answer)
print("\n출처:")
for s in sources:
    print(s)

답변:
청년의 소비습관은 문화예술 관람비 지원을 통해 연간 15~20만 원의 지원을 받으며, 저소득층 청년은 저렴한 임대주택을 이용할 수 있는 혜택이 있다.

출처:
2026_hope_ladder_selected.pdf
2026_hope_ladder_selected.pdf
2026_hope_ladder_selected.pdf
2026_hope_ladder_selected.pdf


In [42]:
def target(inputs: dict):
    q = inputs["question"]
    answer, contexts, sources = run_rag(q)
    return {
        "answer": answer,
        "contexts": contexts
    }

# 6. LangSmith Dataset 생성

In [43]:
from langsmith import Client

client = Client()

dataset_name = "catcher-rag-welfare-eval"

questions = [
    "여성청소년 생리용품 지원의 월 지원금은 얼마인가?",
    "임신 사전건강관리 지원사업에서 여성에게 지원하는 최대 금액은 얼마인가?",
    "저소득 청소년부모 아동양육비 지원의 월 지원금은 얼마인가?",
    "3~5세 유치원 학비 중 국·공립유치원 교육비는 월 얼마인가?",
    "청년내일저축계좌의 정부 매칭 한도는 얼마인가?",
]

ground_truths = [
    "여성청소년 생리용품 지원의 월 지원금은 1만 4,000원이다.",
    "임신 사전건강관리 지원사업에서 여성에게 지원하는 최대 금액은 13만 원이다.",
    "저소득 청소년부모 아동양육비 지원의 월 지원금은 25만 원이다.",
    "3~5세 유치원 학비 중 국·공립유치원 교육비는 월 10만 원이다.",
    "청년내일저축계좌의 정부 매칭 한도는 월 최대 30만 원이며, 3년 만기 시 최대 1,440만 원 적립 가능하다.",
]

# 이미 존재하면 그냥 사용, 없으면 새로 생성
existing = [d for d in client.list_datasets() if d.name == dataset_name]

if existing:
    dataset = existing[0]
    print(f"기존 dataset 사용: {dataset.name}")
else:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Catcher LLM 복지정책 RAG 평가 데이터셋"
    )
    for q, gt in zip(questions, ground_truths):
        client.create_example(
            inputs={"question": q},
            outputs={"ground_truth": gt},
            dataset_id=dataset.id
        )
    print(f"새 dataset 생성 완료: {dataset.name} ({len(questions)}개 예시)")

기존 dataset 사용: catcher-rag-welfare-eval


# 7. Evaluator 정의

In [44]:
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Evaluator 1: 정답 일치도 (0~1)
def correctness_evaluator(run, example):
    answer = run.outputs["answer"]
    ground_truth = example.outputs["ground_truth"]

    prompt = f"""
다음 답변이 정답과 얼마나 일치하는지 0~1 점수로 평가해줘.
핵심 수치(금액, 기간 등)가 맞으면 높게, 틀리면 낮게.

정답: {ground_truth}
답변: {answer}

숫자 하나만 출력해.
"""
    score = judge_llm.invoke(prompt).content.strip()
    return {"key": "correctness", "score": float(score)}


# Evaluator 2: 문서 충실성 — 문서 내용만 사용했는가 (0~1)
def faithfulness_evaluator(run, example):
    answer = run.outputs["answer"]
    contexts = run.outputs["contexts"]

    prompt = f"""
아래 답변이 문서에 있는 내용만 사용했는지 평가해줘.
문서에 없는 정보를 추가했으면 낮게, 문서 내용만 사용했으면 높게.
0~1 숫자 하나만 출력해.

문서:
{contexts}

답변: {answer}
"""
    score = judge_llm.invoke(prompt).content.strip()
    return {"key": "faithfulness", "score": float(score)}


# Evaluator 3: 금액 포함 여부 (heuristic)
def contains_amount_evaluator(run, example):
    import re
    answer = run.outputs["answer"]
    has_amount = bool(re.search(r'\d[\d,]*\s*(원|만원|만\s*원)', answer))
    return {"key": "contains_amount", "score": 1 if has_amount else 0}


print("evaluator 3개 정의 완료")

evaluator 3개 정의 완료


# 8. evaluate() 실행 → LangSmith 반영

In [45]:
from langsmith.evaluation import evaluate

evaluate(
    target,
    data=dataset_name,
    evaluators=[
        correctness_evaluator,
        faithfulness_evaluator,
        contains_amount_evaluator,
    ],
    experiment_prefix="welfare-rag-v1"
)

View the evaluation results for experiment: 'welfare-rag-v1-eb331c49' at:
https://smith.langchain.com/o/f278f778-c6fd-420a-a653-490d4eb37548/datasets/f2a7ff57-5d65-4102-adf0-dcf638c187b0/compare?selectedSessions=14c6898e-65af-46e8-9b80-05ae088fee51




5it [00:23,  4.62s/it]


,inputs.question,outputs.answer,outputs.contexts,error,reference.ground_truth,feedback.correctness,feedback.faithfulness,feedback.contains_amount,execution_time,example_id,id
0,청년내일저축계좌의 정부 매칭 한도는 얼마인가?,청년내일저축계좌의 정부 매칭 한도는 문서에 명시되어 있지 않습니다.,[생애주기별 국민생활 서비스 - 청년·대학생 090\n주거안정장학금 한국장학재단\n...,None,"청년내일저축계좌의 정부 매칭 한도는 월 최대 30만 원이며, 3년 만기 시 최대 1...",0.0,0.0,0,2.885951,708a8b91-ea76-4be6-af6f-3db8c7b107f8,019ddcf2-b7f1-7200-99af-acc1305a1963
1,3~5세 유치원 학비 중 국·공립유치원 교육비는 월 얼마인가?,국·공립유치원 교육비는 월 10만 원입니다.,[모두의 정책 K-희망사다리 2026 061\n3~5세 유치원 학비 0079에듀콜\...,None,3~5세 유치원 학비 중 국·공립유치원 교육비는 월 10만 원이다.,1.0,1.0,1,2.460033,00656b2d-6cfc-40e6-bd7a-f39e1803e7e7,019ddcf2-c938-73d2-9106-8fbde7efd1ce
2,저소득 청소년부모 아동양육비 지원의 월 지원금은 얼마인가?,저소득 청소년부모 아동양육비 지원의 월 지원금은 자녀 1인당 25만 원입니다.,[따뜻한 동행 모두가 행복한 사회 - 숨은 정부지원금 찾기 048\n저소득 청소년부...,None,저소득 청소년부모 아동양육비 지원의 월 지원금은 25만 원이다.,1.0,1.0,1,2.042856,af3c89d0-7860-498b-94c8-53e6fa7d048c,019ddcf2-dc29-7b41-90f9-6c25e09f6623
3,임신 사전건강관리 지원사업에서 여성에게 지원하는 최대 금액은 얼마인가?,임신 사전건강관리 지원사업에서 여성에게 지원하는 최대 금액은 13만 원입니다.,[모두의 정책 K-희망사다리 2026 043\n임신 사전건강관리 보건복지상담센터\n...,None,임신 사전건강관리 지원사업에서 여성에게 지원하는 최대 금액은 13만 원이다.,1.0,1.0,1,2.250697,ba18c18b-7ed6-4e34-a09d-65366a7b2d0c,019ddcf2-edaa-7b63-bace-4d63ef2a07ab
4,여성청소년 생리용품 지원의 월 지원금은 얼마인가?,"여성청소년 생리용품 지원의 월 지원금은 1만 4,000원입니다.",[모두의 정책 K-희망사다리 2026 037\n여성청소년 성평등가족부 청소년정책과\...,None,"여성청소년 생리용품 지원의 월 지원금은 1만 4,000원이다.",1.0,1.0,1,2.070497,e7bef046-8f42-44fe-8a3f-7f0d699d649f,019ddcf2-fc43-7d83-8091-e2e2171c6bfe
